# Unit 5: Speech Recognition (ASR) - Hands-on Exercise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

**Course**: [HuggingFace Audio Transformers Course - Unit 5](https://huggingface.co/learn/audio-course/chapter5)

## Certificate Exercise Requirements

- Fine-tune `openai/whisper-tiny` on `PolyAI/minds14` (en-US subset)
- Use first **450 examples for training**, rest for evaluation
- Set `num_proc=1` when preprocessing with `.map`
- Report `wer` and `wer_ortho` as decimals (not percentages)
- Achieve normalized WER (`wer`) **< 0.37**
- Push model to Hub with required kwargs

## Status: TODO

## Setup

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate librosa soundfile

from huggingface_hub import notebook_login
notebook_login()

## 1. Load and Prepare the Dataset

In [ ]:
from datasets import load_dataset, Audio

minds14 = load_dataset("PolyAI/minds14", name="en-US", split="train", trust_remote_code=True)
print(f"Total examples: {len(minds14)}")
print(f"Features: {minds14.features}")

In [ ]:
# Split: first 450 for training, rest for evaluation (as required)
minds14_train = minds14.select(range(450))
minds14_eval = minds14.select(range(450, len(minds14)))

print(f"Train: {len(minds14_train)} examples")
print(f"Eval: {len(minds14_eval)} examples")

In [ ]:
import IPython.display as ipd

# Listen to a sample
sample = minds14_train[0]
print(f"Transcription: {sample['transcription']}")
print(f"Sampling rate: {sample['audio']['sampling_rate']}")
ipd.Audio(sample['audio']['array'], rate=sample['audio']['sampling_rate'])

## 2. Load Whisper Components

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

model_id = "openai/whisper-tiny"

feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
tokenizer = WhisperTokenizer.from_pretrained(model_id, language="english", task="transcribe")
processor = WhisperProcessor.from_pretrained(model_id, language="english", task="transcribe")

print(f"Feature extractor sampling rate: {feature_extractor.sampling_rate}")

In [ ]:
# Resample to 16kHz
minds14_train = minds14_train.cast_column("audio", Audio(sampling_rate=16000))
minds14_eval = minds14_eval.cast_column("audio", Audio(sampling_rate=16000))

## 3. Preprocess the Data

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    
    # Compute log-mel spectrogram input features
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    
    # Encode target text to label ids
    batch["labels"] = tokenizer(batch["transcription"]).input_ids
    return batch

# IMPORTANT: num_proc=1 as required by the exercise
minds14_train = minds14_train.map(prepare_dataset, remove_columns=minds14_train.column_names, num_proc=1)
minds14_eval = minds14_eval.map(prepare_dataset, remove_columns=minds14_eval.column_names, num_proc=1)

print(f"Train features: {minds14_train.column_names}")
print(f"Eval features: {minds14_eval.column_names}")

## 4. Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Extract input features and convert to tensor batch
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Get tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 to ignore in loss
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove BOS token if it was appended during tokenization
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 5. Evaluation Metrics

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # Decode predictions and labels
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Compute orthographic WER (no normalization)
    wer_ortho = wer_metric.compute(predictions=pred_str, references=label_str)

    # Compute normalized WER
    pred_str_norm = [tokenizer._normalize(text) for text in pred_str]
    label_str_norm = [tokenizer._normalize(text) for text in label_str]
    # Filter out empty references after normalization
    pred_str_norm_filtered = [
        pred_str_norm[i] for i in range(len(label_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm_filtered = [
        label_str_norm[i] for i in range(len(label_str_norm)) if len(label_str_norm[i]) > 0
    ]
    wer = wer_metric.compute(predictions=pred_str_norm_filtered, references=label_str_norm_filtered)

    # Return as decimals, NOT percentages
    return {"wer": wer, "wer_ortho": wer_ortho}

## 6. Load Model and Train

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Configure generation settings
model.generation_config.language = "english"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-tiny-minds14-en-us",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=minds14_train,
    eval_dataset=minds14_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

## 7. Evaluate

In [ ]:
eval_results = trainer.evaluate()

print(f"\nNormalized WER: {eval_results['eval_wer']:.4f}")
print(f"Orthographic WER: {eval_results['eval_wer_ortho']:.4f}")
print(f"Target: wer < 0.37")
print(f"{'PASSED' if eval_results['eval_wer'] < 0.37 else 'NEEDS IMPROVEMENT'}")

## 8. Push to Hub (Required for Certificate)

In [ ]:
kwargs = {
    "dataset_tags": "PolyAI/minds14",
    "finetuned_from": "openai/whisper-tiny",
    "tasks": "automatic-speech-recognition",
}

trainer.push_to_hub(**kwargs)

## 9. Build a Quick Demo

In [ ]:
from transformers import pipeline

# Test the fine-tuned model
asr = pipeline("automatic-speech-recognition", model="./whisper-tiny-minds14-en-us")

# Test on a sample from the eval set
minds14_eval_raw = load_dataset("PolyAI/minds14", name="en-US", split="train", trust_remote_code=True)
test_sample = minds14_eval_raw[450]
result = asr(test_sample["audio"]["array"])

print(f"Ground truth: {test_sample['transcription']}")
print(f"Prediction:   {result['text']}")